# miniLLM 49M — SFT : 4 variantes comparées (A / B / C / D)

Chaque run dure ~6 min sur T4 et écrit dans son propre dossier (`checkpoints/sft_A` … `sft_D`) : **rien n'est écrasé**.
Hyperparamètres identiques partout (2 époques, LR 1e-4 → 1e-5) : seules les **données** changent, une chose à la fois.

| run | données |
|---|---|
| **A** | recette du SFT v3 d'origine (témoin) |
| **B** | A + `synthetic_plus` allégé (logique, intrus, langue, calendrier + ~1 000 calculs ; sans définitions) |
| **C** | B + filtre qualité (refus, boucles, échos) + persona fixée à 6 % du train |
| **D** | C + réponses GPT-4 FR **filtrées** (teste cette source proprement) |

On compare **final.pt et best.pt** de chaque run : `best.pt` (choisi par val_loss) tombe en milieu d'entraînement et a donc moins mémorisé la persona que `final.pt`.

**Utilisation :** Exécution → type d'exécution → GPU (T4), puis **Tout exécuter**. Si la session coupe, relance : chaque étape reprend où elle s'est arrêtée.

In [ ]:
# ── 1) Drive, code, garde-fou, découpage des nombres par le tokenizer ──
import os, sys, glob, shutil, json
from google.colab import drive
drive.mount("/content/drive")

cands = (["/content/drive/MyDrive/MiniLLM_v2"] + glob.glob("/content/drive/MyDrive/*/MiniLLM_v2")
         + glob.glob("/content/drive/Shareddrives/*/MiniLLM_v2"))
BASE = next((c for c in cands if os.path.isdir(f"{c}/data")), None)
if BASE is None:
    raise SystemExit("Dossier MiniLLM_v2 introuvable (Drive > Partagés > appui long > Ajouter un raccourci > Mon Drive).")
DATA, PT_OUT, CKPT = f"{BASE}/data", f"{BASE}/checkpoints/pretrain", f"{BASE}/checkpoints"
CODE_DIR, LOCAL = "/content/minillm_v2", "/content/minillm_local"

!pip -q install tokenizers datasets
if os.path.exists(f"{CODE_DIR}/.git"):
    !git -C {CODE_DIR} fetch -q origin v2.1.1 && git -C {CODE_DIR} checkout -q v2.1.1 && git -C {CODE_DIR} pull -q --ff-only
else:
    !git clone -q -b v2.1.1 https://github.com/bono-p/minillm_v2.git {CODE_DIR}
os.chdir(CODE_DIR); sys.path.insert(0, CODE_DIR)

import torch
b = torch.load(f"{PT_OUT}/best.pt", map_location="cpu", weights_only=False)
print(f"Base du SFT : pré-entraînement it={b['iter']} | val_loss={b['val_loss']:.4f}")
if int(b["iter"]) < 49_000:
    raise SystemExit("Le pré-entraînement n'est pas terminé (it < 49 000).")
del b

# Comment le tokenizer découpe les nombres ? (explique pourquoi l'arithmétique ne s'apprend pas)
from tokenizers import Tokenizer
_t = Tokenizer.from_file(f"{DATA}/tokenizer.json")
print("\nDécoupage des nombres par le tokenizer :")
for s in ["26 + 43 = 69", "12 × 12 = 144", "1000 - 275 = 725"]:
    print(f"  {s:<18} -> {_t.encode(s).tokens}")


In [ ]:
# ── 2) Données des 4 variantes (dossiers data/sft_A … sft_D ; ignorées si déjà prêtes) ──
COMMON = f"--tokenizer {DATA}/tokenizer.json --oasst 3000 --piaf 2000 --evol 0 --sharegpt 0 --persona personnalite.jsonl"
RUNS = {
    "A": "--alpaca 15000 --alpaca_gpt4 0 --plus 0 --persona_repeat 8",
    "B": "--alpaca 15000 --alpaca_gpt4 0 --plus 1 --plus_light 1 --persona_repeat 8",
    "C": "--alpaca 15000 --alpaca_gpt4 0 --plus 1 --plus_light 1 --quality_filter 1 --persona_share 0.06",
    "D": "--alpaca 15000 --alpaca_gpt4 8000 --plus 1 --plus_light 1 --quality_filter 1 --persona_share 0.06",
}
for name, flags in RUNS.items():
    out = f"{DATA}/sft_{name}"
    if os.path.exists(f"{out}/meta.json"):
        print(f"✓ données {name} déjà prêtes :", json.load(open(f"{out}/meta.json"))["sources"])
        continue
    print("\n" + "=" * 70 + f"\nDonnées {name}\n" + "=" * 70)
    !python sft_data.py {COMMON} {flags} --out {out}
    srcs = json.load(open(f"{out}/meta.json"))["sources"]
    if name == "D" and "alpaca_gpt4_fr" not in srcs:
        print("\n⚠️  GPT-4 FR n'a rien chargé : D est identique à C (ne le compte pas comme une variante).")


In [ ]:
# ── 3) Entraînement A → D (~6 min chacun ; relance possible : chaque run reprend où il en est) ──
for name in RUNS:
    local = f"{LOCAL}/sft_{name}"
    if os.path.exists(local):
        shutil.rmtree(local)
    shutil.copytree(f"{DATA}/sft_{name}", local)
    print("\n" + "=" * 70 + f"\nEntraînement {name}\n" + "=" * 70)
    !python train.py --mode sft --data_dir {local} --out_dir {CKPT}/sft_{name} --init_from {PT_OUT}/best.pt \
        --epochs 2 --lr 1e-4 --min_lr 1e-5 --weight_decay 0.1 --dropout 0.1 --warmup_iters 100 --patience 0 \
        --eval_every 250 --eval_every_dense 100 --eval_dense_until 1500 --save_every 500


In [ ]:
# ── 4) Tableau comparatif : mêmes mesures, même jeu de validation (celui de A) pour tous ──
WITH_BEST = True        # False = ne compare que final.pt (2× plus rapide)
specs = []
for name in RUNS:
    specs += ["--ckpt", f"{name}_final={CKPT}/sft_{name}/final.pt"]
    if WITH_BEST:
        specs += ["--ckpt", f"{name}_best={CKPT}/sft_{name}/best.pt"]
os.makedirs(f"{BASE}/evals", exist_ok=True)
SPECS = " ".join(specs)
!python compare_sft.py {SPECS} --val_dir {DATA}/sft_A --out_dir {BASE}/evals --qa_n 300


**À m'envoyer :** le tableau final (et les lignes `retirés : {...}` de la cellule 2 si tu les as).
Les réponses aux 337 questions de chaque modèle sont sauvegardées dans `MiniLLM_v2/evals/openqa_<nom>.txt`.

Lecture du tableau — `pers.EM` haut : identité mémorisée · `basics` haut : faits conservés · `distinct` bas : réponses toutes pareilles (effondrement) · `refus` / `boucles` : défauts à éviter.